____
### 1. Imports

In [1]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Normal

____
### 2. Environment

Creating a custom environment to simulate the following condtions:
1. 30 day period
2. Required to sell inventory of 100 units
3. Unit cost of each unit set at $5
4. Agents in the environment are required to maximise profit in the 30 day period


Methods (1 - 3 are required for all environments):
1. `__init__()` → setup
2. `reset()` → start a new episode
3. `step(action)` → apply an action
4. `demand(price)` → calculates the demand given a price

Attributes:
1. observation space, 5 numbers as an array [Leftover Stock, Days Left, Last known demand]
2. action space (price of good, number over a continuouse range from 5 to 50)
3. max steps (days of simulation)
4. cost (cost incurred to obtain 1 unit)


In [9]:
class DynamicPricingEnv(gym.Env):
    def __init__(self):
        self.max_steps = 30
        self.max_inventory = 100
        self.cost = 5.0
        self.observation_space = gym.spaces.Box( #observation is given by [inventory, days left, number of units sold last]
            low=np.array([0, 0, 0]),
            high=np.array([self.max_inventory, self.max_steps, self.max_inventory]),
            dtype=np.float32
        )
        self.action_space = gym.spaces.Box(low=5.0, high=50.0, shape=(1,), dtype=np.float32)
        #action is a discrete number between 5 to 50
        self.latest = np.array([self.max_inventory, self.max_steps, 0], dtype=np.float32)

    def reset(self, seed=None, options=None): #resets the values in the state
        self.inventory = self.max_inventory 
        self.step_count = 0
        self.last_demand = 0
        self.latest = self.obs()
        return self.obs(), {} #extra info dictionary is empty

    def step(self, action):
        price = float(action[0]) #obtain the price as a float
        demand = self.demand(price) #calculating demand from the price set
        units_sold = min(demand, self.inventory) #sell out 

        self.inventory -= units_sold #updated inventory
        self.step_count += 1 # step + 1
        self.last_demand = units_sold # updating last known demand

        reward = (price - self.cost) * units_sold #reward is the profit generated from that round
        terminated = self.inventory <= 0
        truncated = self.step_count >= self.max_steps
        self.latest = self.obs()
        return self.obs(), reward, terminated, truncated, {}

    def obs(self): #to obtain observation of the current state
        return np.array([self.inventory, self.max_steps - self.step_count,
                         self.last_demand], dtype=np.float32)

    def get_latest(self):
        obs = self.latest
        print(f"Leftover Stock: {obs[0]} units, Days Left {obs[1]}, Sold Units: {obs[2]}")

    def demand(self, price):
        base = 40
        sensitivity = 1.2
        noise = np.random.normal(0, 3) #noise is random number from 0 to 3
        return int(max(0, base - sensitivity * price + noise))

____
#### 2.1 Exploring the environment

In [3]:
env = DynamicPricingEnv()

In [10]:
obs, info = env.reset()
print(f"Observation space representing: [stock left, days left, last known sold]: {obs}")
print(f"Extra info dictrionary: {info}")

obs, reward, terminated, truncated, info = env.step(10)
env.get_latest()
print(reward)
print(f"{terminated}, {truncated}")

Observation space representing: [stock left, days left, last known sold]: [100.  30.   0.]
Extra info dictrionary: {}
Leftover Stock: 72.0 units, Days Left 29.0, Sold Units: 28.0
140.0
False, False


____
### 3. Deciding which model to use 
- A standard Q-table cannot be used as the action space (price of good) is a continuous number instead of a discrete number

#### 3.1 Proximal Policy Optimization (PPO) Model
- a policy gradient method which directly learns "Given this state, what price should I output"
- the "Proximal" part of the PPO model adjusts the actions in small amounts to find the optimal policy 
- therefore, PPO Models limits how drastically the policy changes each update to prevent unstable training

##### PPO Architecture
- PPO uses 2 networks
1. Actor Network
    - Outputs the pricing policy
    - Given a state, feeds into a neutal network and outputs a pricing distribution
    - `state → neural network → price distribution`
    - The agent then samples prices around that range

2. Critic Network
    - Estimates the future rewards
    - Asks "How profitable is this situation" and helps the Actor Network improve

##### PPO Flow
`Observe market → Choose price → Simulate customer response → Get profit reward → Update pricing policy slightly`


#### 3.2 Twin Delayed Deep Deterministic Policy Gradient (TD3)
- Designed specefically for continuous action space, precise control and stable deep Q-learning
- instead learning "what action should I take?" the model learns "how good is a particular action"

##### TD3 Architecture
- TD3 uses 3 networks
1. Actor Network
    - Outputs a distribution over actions
    - `state → distribution`
    - Samples from the distribution to create exploration

2. Critic Network (2 critic networks)
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` (same as TD3)

3. Replay Buffer
    - To store experiences and reuse them, making the SAC highly sample efficient
    - Allow SAC to learn from past experiences repeatedly

4. Target Networks

##### TD3 Flow
- `Observe State → Actor suggests an action → Twin Critic Networks evaluate long term reward of the actions → Lower Q-value generated from the 2 networks is used`
- Therefore, the lower Q-value is used to train the critics and the actor (actor updated occassionally)



#### 3.3 Soft Actor-Critic (SAC)
- Considered one of the strongest RL algorithms for continuous control
- Combines actor-critic learning, entropy maximisation and off-policy training
- Tries to maximise both `Reward` and `Exploration` instead of only `profit`
- Entropy refers to the epsilon (randomness of the actions) therefore it encourages the agent to keep exploring pricing options, preventing the model from becoming too deterministic 


##### SAC Architecture
- SAC uses 3 networks
1. Actor Network
    - Outputs the exact price
    - `state → price`

2. Critic Network (2 critic networks)
    - The "Twin" part is in refernce to the 2 critic networks
    - Each critic network estimates a Q-value, which represents the total future reward if an action is taken in this state, ie `Q(state, action)` 

##### SAC Flow
- `Observe State → Sample action from policy distribution → receive reward → update critic → update actor → encourage exploration through entropy bonus`
- Reward is evaluated as `total reward = reward + entropy bonus` and to encourage exploration

____
### 4. Training the model 
- In the spirit of learning, I will be training a PPO model, a TD3 model and a SAC model

#### 4.1 PPO Model
- Actor and Critic networks are first created

In [11]:
class ActorCritic(nn.Module): # neural network defined as a PyTorch module, inheriting from nn.module
    def __init__(self, obs_dim, action_dim):
        super().__init__() #initialises the parent nn.Module class
        # self.backbone acts as a shared extractor used by both the actor and critic network
        self.backbone = nn.Sequential( #nn.Sequential runs the layers in order, linear -> Tanh -> linear -> tanh
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
        )
        self.actor_mean = nn.Linear(64, action_dim) 
        self.log_std = nn.Parameter(torch.zeros(action_dim)) 
        self.critic = nn.Linear(64, 1)

    def forward(self, obs): #needs to be overridden
        features = self.backbone(obs) #vector of 64 values
        mean = self.actor_mean(features) #obtain the action 
        std = self.log_std.exp().expand_as(mean) #obtain the std dev and fits the shape with with mean 
        critic_val = self.critic(features) #obtain the critic value
        return mean, std, critic_val
    
    def get_action(self, obs): #creating action based off the network
        mean, std, value = self.forward(obs) #calling forward to obtain values from actor and critic network
        dist = Normal(mean, std) #creates normal distribution
        action = dist.sample() #samples the distribution
        log_prob = dist.log_prob(action).sum(dim=-1) #obtains sum of log distribution (exp below)
        return action, log_prob, value.squeeze(-1) #converts the value into a scalar

    def evaluate(self, obs, action):
        mean, std, value = self.forward(obs)
        dist = Normal(mean, std)
        log_prob = dist.log_prob(action).sum(dim=-1)
        entropy = dist.entropy().sum(dim=-1) #calculates the entropy of the normal distribution, how random or uncertain the distribution is 
        # entropy is summed up over the last dimension
        return log_prob, value.squeeze(-1), entropy

##### 4.11 Explanation of Code:
```python
self.backbone = nn.Sequential( 
            nn.Linear(obs_dim, 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh()
        )
```
- `nn.sequential()` ensures that the following layers run in sequence
- `nn.Linear(obs_dim, 64)` -> y = Wx + b
    - x is a `3x1 vector` since the observation is a 3-dimensional vector with 3 values
    - W is a `64x3 matrix` such that the product `Wx` is a 64-dimensional vector with 64 values
    - b is a `64x1 vector` bias function that weighs down each value inside the produce `Wx`
    - y is the resultant `64x1 vector`, representing the first layer of the neural network, where each number of the vector corresponds to a neuron
- `nn.Tanh()` -> y = tanh(x)
    - x is a `64 x 1 vector` that is the result from the linear vector
    - y is the resultant `64 x 1 vector` from applying the tanh() function on every single value in the original vector
    - this function introduces non-linearlity and squashes every value to be within the range (-1, 1)
    - the introduction of non-linearity allows the model to learn non-linear behaviours
- `nn.Linear(64, 64)` -> y = Wx + b
    - uses the non-linear outputs from the previous layers but turns them into more sophisticated outputs using weights and biases
- `nn.Tanh()` -> y = tanh(x)
    - squishes all values to within the range (-1, 1) and introduces non-linearity to the outputs of the previous layer

```python
self.actor_mean = nn.Linear(64, action_dim) 
self.log_std = nn.Parameter(torch.zeros(action_dim))
self.critic = nn.Linear(64, 1)
```
- `self.actor_mean = nn.Linear(64, action_dim)`
    - Makes use of the 64 outputs from the backbone to derive the outputs in the action dimension (mean of the price)
- `self.log_std = nn.Parameter(torch.zeroes(action_dim))`
    - std deviation measures the randomness of the distributionm, log(std) is used so that the value is not -ve, later converted using `exp()`
    - wrapping it in `nn.Parameter()` means that the value should be learning during training
    - this makes it such that the PPO model learns during training what is the appropriate level of exploration
- `self.critic = nn.Linear()`
    - Makes use of the 64 outputs from the backbone to derive 1 value, which represents the future reward expected from the current state

- `log_prob = dist.log_prob(action).sum(dim=-1)`
    - `dist.log_prob(action)` obtains the log_probability of the action within the distribution
    - `.sum(dim=-1)` sums it along the last dimension
    - instad of multiplying individual probabilities, it adds up `log(prob)` instead 

##### 4.12 Over-Arching View:
- observation (vector of 3 values) -> self.backbone(vector of 64 values) 
- The observation in the form of 64 values is then fed into the actor network and critic network
- ie `self.backbone(vector of 64 values) -> action (1 value)` and `self.backbone(vector of 64 values) -> future reward expected (1 value)`

##### 4.13 Creating RolloutBuffer
- Acts as a container to store past experiences

In [12]:
class RolloutBuffer:
    def __init__(self):
        self.clear()

    def clear(self): #resets all lists to empty
        self.obs, self.actions, self.log_probs = [], [], []
        self.rewards, self.values, self.dones  = [], [], []

    def add(self, obs, action, log_prob, reward, value, done): #appends one round of experiences
        self.obs.append(obs)
        self.actions.append(action)
        self.log_probs.append(log_prob)
        self.rewards.append(reward)
        self.values.append(value)
        self.dones.append(done)

    def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95): #compute generalized advantage est
        advantages = [] 
        gae = 0.0
        values = self.values + [last_value] # adds one extra value to compute next-step difference
        for t in reversed(range(len(self.rewards))):
            delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
            gae   = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
            advantages.insert(0, gae)
        returns = [adv + val for adv, val in zip(advantages, self.values)]
        return advantages, returns

    def to_tensors(self, advantages, returns, device):
        obs = torch.tensor(np.array(self.obs), dtype=torch.float32).to(device)
        actions = torch.stack(self.actions).to(device)
        log_probs = torch.stack(self.log_probs).to(device)
        advantages = torch.tensor(advantages, dtype=torch.float32).to(device)
        returns = torch.tensor(returns, dtype=torch.float32).to(device)
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        return obs, actions, log_probs, advantages, returns

##### 4.14 Explanation of code

```python
def compute_returns(self, last_value, gamma=0.99, gae_lambda=0.95):
    advantages = [] 
    gae = 0.0
    values = self.values + [last_value] 
    for t in reversed(range(len(self.rewards))): #iterate backwards according to the number of rewards
        delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]
        gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae
        advantages.insert(0, gae)
    returns = [adv + val for adv, val in zip(advantages, self.values)]
    return advantages, returns
```
- `compute_returns()` is meant to compute the advantages ("How good was this action") and the returns ("how good was this state overall")
- `values = self.values + [last_value]`
    - appends the critic network's estimate for the final state
- `delta = self.rewards[t] + gamma * values[t + 1] * (1 - self.dones[t]) - values[t]`
- `delta = (actual outcome) - (expected outcome)`, measures outcome of the action compared to expected outcome
    - delta < 0, worse // delta > 0, better
    - `self.rewards[t]` is the immediate reward
    - `gamma * values[t + 1] * (1 - self.dones[t])` is the future term which estimates the future rewards
    - `gamma` is the discount factor, measures how much the model cares about future rewards
    - `values[t + 1]` is the predicted future value, the critic's estimate of future reward
    - `(1 - self.dones[t])` is a flipper that prevents bootstrapping (using own predictions to estimate future outcomes)
    - when the episode is done, `self.dones[t]` = 1 and the future term becomes zero
    - `values[t]` is baseline expectation before seeing reward
- `gae = delta + gamma * gae_lambda * (1 - self.dones[t]) * gae`
- `gae = current TD + discounted future gae`, acts as a memory of past TD errors, representing accumulated advantage from future steps
    - it recursively accumulates future TD errors using `gamma * lambda`
    - this creates a smooth estimate of how good an action was over time
    - `done[t] - 1` acts as a stopper to prevent recursion from leaking across episodes

- `advantages.insert(0, gae)`
    - inserts the calculated gae at the `front` of the advantages array as it is iterated through in reverse

- `returns = [adv + val for adv, val in zip(advantages, self.values)]`
- equivalent to doing:
    ```python
    returns = []
    for adv, val in zip(advantages, self.values):
        r = adv + val
        returns.append(r)
    ```
    - for each step, returns = advantage + value estimate
    - adding it to the array saved as returns 


##### 4.15 Instantiate functions to update PPO and train the model

In [13]:
def ppo_update(model, optimizer, obs, actions, old_log_probs,
               advantages, returns, clip_range=0.2, ent_coef=0.01,
               vf_coef=0.5, n_epochs=10, batch_size=64):
    total_steps = obs.shape[0]
    for _ in range(n_epochs):
        indices = torch.randperm(total_steps)
        for start in range(0, total_steps, batch_size):
            idx = indices[start : start + batch_size]
            # re-evaluate actions under the current policy
            new_log_probs, values, entropy = model.evaluate(obs[idx], actions[idx])

            # ratio for importance sampling
            ratio = (new_log_probs - old_log_probs[idx]).exp()

            # clipped surrogate objective
            adv = advantages[idx]
            policy_loss = -torch.min(
                ratio * adv,
                torch.clamp(ratio, 1 - clip_range, 1 + clip_range) * adv
            ).mean()

            # value function loss
            value_loss = nn.functional.mse_loss(values, returns[idx])

            # entropy bonus — encourages exploration
            entropy_loss = -entropy.mean()

            # combined loss
            loss = policy_loss + vf_coef * value_loss + ent_coef * entropy_loss

            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
            optimizer.step()


# ── 5. Training Loop ──────────────────────────────────────────────
def train(total_timesteps=200_000, n_steps=512):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on: {device}")

    env    = DynamicPricingEnv()
    obs_dim    = env.observation_space.shape[0]   # 3
    action_dim = env.action_space.shape[0]        # 1

    model     = ActorCritic(obs_dim, action_dim).to(device)
    optimizer = optim.Adam(model.parameters(), lr=3e-4)
    buffer    = RolloutBuffer()

    obs, _        = env.reset()
    episode_reward = 0
    episode_count  = 0
    timestep       = 0

    while timestep < total_timesteps:
        buffer.clear()

        # ── Collect n_steps of experience ────────────────────────
        for _ in range(n_steps):
            obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)

            with torch.no_grad():
                action, log_prob, value = model.get_action(obs_tensor)

            # clip action to valid price range
            action_np = action.cpu().numpy()[0]
            action_np = np.clip(action_np, 5.0, 50.0)

            next_obs, reward, terminated, truncated, _ = env.step(action_np)
            done = terminated or truncated

            buffer.add(
                obs      = obs,
                action   = action.squeeze(0).cpu(),
                log_prob = log_prob.squeeze(0).cpu(),
                reward   = reward,
                value    = value.squeeze(0).cpu().item(),
                done     = float(done),
            )

            episode_reward += reward
            obs             = next_obs
            timestep       += 1

            if done:
                episode_count += 1
                if episode_count % 20 == 0:
                    print(f"Timestep {timestep:>7} | Episode {episode_count:>4} | "
                          f"Reward: {episode_reward:>8.2f}")
                episode_reward = 0
                obs, _         = env.reset()

        # ── Compute returns and update ────────────────────────────
        with torch.no_grad():
            last_obs    = torch.tensor(obs, dtype=torch.float32).unsqueeze(0).to(device)
            _, _, last_value = model.get_action(last_obs)
            last_value  = last_value.squeeze(0).cpu().item()

        advantages, returns = buffer.compute_returns(last_value)
        obs_t, act_t, lp_t, adv_t, ret_t = buffer.to_tensors(
            advantages, returns, device
        )

        ppo_update(model, optimizer, obs_t, act_t, lp_t, adv_t, ret_t)

    torch.save(model.state_dict(), "ppo_pricing.pth")
    print("Training complete. Model saved to ppo_pricing.pth")
    return model

In [14]:
model = train(total_timesteps=200_000)

Training on: cpu
Timestep      68 | Episode   20 | Reward:     0.00
Timestep     138 | Episode   40 | Reward:     0.00
Timestep     207 | Episode   60 | Reward:     0.00
Timestep     277 | Episode   80 | Reward:     0.00
Timestep     348 | Episode  100 | Reward:     0.00
Timestep     418 | Episode  120 | Reward:     0.00
Timestep     488 | Episode  140 | Reward:     0.00
Timestep     556 | Episode  160 | Reward:     0.00
Timestep     623 | Episode  180 | Reward:     0.00
Timestep     694 | Episode  200 | Reward:     0.00
Timestep     762 | Episode  220 | Reward:     0.00
Timestep     834 | Episode  240 | Reward:     0.00
Timestep     900 | Episode  260 | Reward:     0.00
Timestep     970 | Episode  280 | Reward:     0.00
Timestep    1039 | Episode  300 | Reward:     0.00
Timestep    1102 | Episode  320 | Reward:     0.00
Timestep    1176 | Episode  340 | Reward:     0.00
Timestep    1245 | Episode  360 | Reward:     0.00
Timestep    1313 | Episode  380 | Reward:     0.00
Timestep    13

In [15]:
def evaluate_model(model, n_episodes=50, deterministic=True):
    """Evaluate a trained pricing model over multiple episodes."""
    device = next(model.parameters()).device
    model.eval()

    episode_rewards = []
    ending_inventory = []
    steps_taken = []

    for _ in range(n_episodes):
        env = DynamicPricingEnv()
        obs, _ = env.reset()
        done = False
        total_reward = 0.0

        while not done:
            obs_tensor = torch.tensor(obs, dtype=torch.float32, device=device).unsqueeze(0)

            with torch.no_grad():
                mean, std, _ = model.forward(obs_tensor)
                if deterministic:
                    action = mean
                else:
                    action = Normal(mean, std).sample()

            action_np = action.squeeze(0).detach().cpu().numpy()
            action_np = np.clip(action_np, 5.0, 50.0).astype(np.float32)

            obs, reward, terminated, truncated, _ = env.step(action_np)
            total_reward += reward
            done = terminated or truncated

        episode_rewards.append(total_reward)
        ending_inventory.append(env.inventory)
        steps_taken.append(env.step_count)

    results = {
        "episode_rewards": episode_rewards,
        "mean_reward": float(np.mean(episode_rewards)),
        "std_reward": float(np.std(episode_rewards)),
        "min_reward": float(np.min(episode_rewards)),
        "max_reward": float(np.max(episode_rewards)),
        "mean_ending_inventory": float(np.mean(ending_inventory)),
        "mean_steps": float(np.mean(steps_taken)),
    }

    print(f"Episodes: {n_episodes}")
    print(f"Mean reward: {results['mean_reward']:.2f} +/- {results['std_reward']:.2f}")
    print(f"Reward range: [{results['min_reward']:.2f}, {results['max_reward']:.2f}]")
    print(f"Mean ending inventory: {results['mean_ending_inventory']:.2f}")
    print(f"Mean steps per episode: {results['mean_steps']:.2f}")

    return results


# Example usage:
# eval_results = evaluate_model(model, n_episodes=100, deterministic=True)

In [16]:
eval_results = evaluate_model(model, n_episodes=100, deterministic=True)

Episodes: 100
Mean reward: 238.38 +/- 39.77
Reward range: [156.54, 327.32]
Mean ending inventory: 0.00
Mean steps per episode: 4.00
